# Graph Convolutional Neural Network (train and test separated)

In [34]:
batch_size = 8 
train_files = [
    {"label": [0], "topology_filename": "../FP11_monomer/SYSTEMns.prmtop", "coordinates_filename": "../FP11_monomer/PRODUCTION_0/TRAJ-150ns.nc"},
    {"label": [0], "topology_filename": "../FP11_monomer/SYSTEMns.prmtop", "coordinates_filename": "../FP11_monomer/PRODUCTION_1/TRAJ-150ns.nc"},
    {"label": [1], "topology_filename": "../FP12-TLR4-MDago_monomer/SYSTEMns.prmtop", "coordinates_filename": "../FP12-TLR4-MDago_monomer/PRODUCTION_0/TRAJ-150ns.nc"},
    {"label": [1], "topology_filename": "../FP12-TLR4-MDago_monomer/SYSTEMns.prmtop", "coordinates_filename": "../FP12-TLR4-MDago_monomer/PRODUCTION_1/TRAJ-150ns.nc"}
]
test_files = [
    {"label": [0], "topology_filename": "../FP11_monomer/SYSTEMns.prmtop", "coordinates_filename": "../FP11_monomer/PRODUCTION_2/TRAJ-150ns.nc"},
    {"label": [1], "topology_filename": "../FP12-TLR4-MDago_monomer/SYSTEMns.prmtop", "coordinates_filename": "../FP12-TLR4-MDago_monomer/PRODUCTION_2/TRAJ-150ns.nc"}
]
selection_string = "resid 613:754 and name CA N C"
epochs = 15
test_size = 0.20
random_state = 42
lr=0.001

In [35]:
train_files

[{'label': [0],
  'topology_filename': '../FP11_monomer/SYSTEMns.prmtop',
  'coordinates_filename': '../FP11_monomer/PRODUCTION_0/TRAJ-150ns.nc'},
 {'label': [0],
  'topology_filename': '../FP11_monomer/SYSTEMns.prmtop',
  'coordinates_filename': '../FP11_monomer/PRODUCTION_1/TRAJ-150ns.nc'},
 {'label': [1],
  'topology_filename': '../FP12-TLR4-MDago_monomer/SYSTEMns.prmtop',
  'coordinates_filename': '../FP12-TLR4-MDago_monomer/PRODUCTION_0/TRAJ-150ns.nc'},
 {'label': [1],
  'topology_filename': '../FP12-TLR4-MDago_monomer/SYSTEMns.prmtop',
  'coordinates_filename': '../FP12-TLR4-MDago_monomer/PRODUCTION_1/TRAJ-150ns.nc'}]

In [36]:
test_files

[{'label': [0],
  'topology_filename': '../FP11_monomer/SYSTEMns.prmtop',
  'coordinates_filename': '../FP11_monomer/PRODUCTION_2/TRAJ-150ns.nc'},
 {'label': [1],
  'topology_filename': '../FP12-TLR4-MDago_monomer/SYSTEMns.prmtop',
  'coordinates_filename': '../FP12-TLR4-MDago_monomer/PRODUCTION_2/TRAJ-150ns.nc'}]

In [37]:
import torch
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

from torch.nn import Embedding, CrossEntropyLoss, Linear, ReLU
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GATConv
from torch_geometric.nn import radius_graph
from torch_geometric.utils import scatter
from sklearn.model_selection import train_test_split

import MDAnalysis as mda
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import tqdm
import json

## DATA LOAD

In [38]:
with open('atom-list.json') as f: 
    atom_type_index = json.load(f)
with open('residue-list.json') as f: 
    residue_index = json.load(f)

In [39]:
def trajectory_to_data(u, selection_string, residue_index, atom_type_index, label):
    for fr in u.trajectory:
        x = torch.from_numpy(u.select_atoms(selection_string).positions)
        residues = list(map(lambda x: residue_index[x], u.select_atoms(selection_string).resnames))
        atom_types = list(map(lambda x: atom_type_index[x], u.select_atoms(selection_string).atoms.types.tolist()))
        d = Data(pos=x, res=torch.tensor(residues, dtype=torch.long), types=torch.tensor(atom_types, dtype=torch.long), label=torch.tensor(label, dtype=torch.long))
        yield d

In [40]:
train_trj = []

for file in train_files:

    u = mda.Universe(file['topology_filename'], file['coordinates_filename'])
    train_trj += list(trajectory_to_data(u, selection_string, residue_index, atom_type_index, file['label']))
    print(file['coordinates_filename']) # to check all files are read correctly
    print(len(u.trajectory)) # to check there are 1500 frames per file
    
train_trj[:5]

../FP11_monomer/PRODUCTION_0/TRAJ-150ns.nc
1500
../FP11_monomer/PRODUCTION_1/TRAJ-150ns.nc
1500
../FP12-TLR4-MDago_monomer/PRODUCTION_0/TRAJ-150ns.nc
1500
../FP12-TLR4-MDago_monomer/PRODUCTION_1/TRAJ-150ns.nc
1500


[Data(pos=[426, 3], res=[426], types=[426], label=[1]),
 Data(pos=[426, 3], res=[426], types=[426], label=[1]),
 Data(pos=[426, 3], res=[426], types=[426], label=[1]),
 Data(pos=[426, 3], res=[426], types=[426], label=[1]),
 Data(pos=[426, 3], res=[426], types=[426], label=[1])]

In [41]:
print(len(train_trj))

6000


In [42]:
test_trj = []

for file in test_files:

    u = mda.Universe(file['topology_filename'], file['coordinates_filename'])
    test_trj += list(trajectory_to_data(u, selection_string, residue_index, atom_type_index, file['label']))
    print(file['coordinates_filename']) # to check all files are read correctly
    print(len(u.trajectory)) # to check there are 1500 frames per file
    
test_trj[:5]

../FP11_monomer/PRODUCTION_2/TRAJ-150ns.nc
1500
../FP12-TLR4-MDago_monomer/PRODUCTION_2/TRAJ-150ns.nc
1500


[Data(pos=[426, 3], res=[426], types=[426], label=[1]),
 Data(pos=[426, 3], res=[426], types=[426], label=[1]),
 Data(pos=[426, 3], res=[426], types=[426], label=[1]),
 Data(pos=[426, 3], res=[426], types=[426], label=[1]),
 Data(pos=[426, 3], res=[426], types=[426], label=[1])]

In [43]:
print(len(test_trj))

3000


## MODEL

In [44]:
class GCN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.rg = radius_graph
        self.embed_res = Embedding(num_embeddings=23, embedding_dim=128)
        self.embed_type = Embedding(num_embeddings=33, embedding_dim=128)
        #
        self.conv1_res = GCNConv(128, 64)
        self.relu1_res = ReLU()
        self.conv2_res = GCNConv(64, 32)
        self.relu2_res = ReLU()
        self.conv3_res = GATConv(32, 16)
        # 
        self.conv1_type = GCNConv(128, 64)
        self.relu1_type = ReLU()
        self.conv2_type = GCNConv(64, 32)
        self.relu2_type = ReLU()
        self.conv3_type = GCNConv(32, 16)
        # 
        self.linear1 = Linear(32, 2)

    def forward(self, data):
        pos, res, type, batch = data.pos, data.res, data.types, data.batch
        edge_index = radius_graph(pos, r=5.0, batch=batch)
        #
        z_res = self.embed_res(res)
        z_res = self.conv1_res(z_res, edge_index)
        z_res = self.relu1_res(z_res)
        z_res = self.conv2_res(z_res, edge_index)
        z_res = self.relu2_res(z_res)
        z_res = self.conv3_res(z_res, edge_index)
        #
        z_type = self.embed_type(type)
        z_type = self.conv1_type(z_type, edge_index)
        z_type = self.relu1_type(z_type)
        z_type = self.conv2_type(z_type, edge_index)
        z_type = self.relu2_type(z_type)
        z_type = self.conv3_type(z_type, edge_index)
        #
        z_res = scatter(z_res, data.batch, dim=0, reduce='mean')
        z_type = scatter(z_type, data.batch, dim=0, reduce='mean')
        z = torch.cat([z_res, z_type], dim=1)
        z = self.linear1(z)
        z = F.softmax(z, dim=1)
        return z

In [45]:
gcn = GCN()
loss = CrossEntropyLoss()
optimizer = torch.optim.Adam(gcn.parameters(), lr=lr, weight_decay=5e-4)
train_loader = DataLoader(train_trj, batch_size=batch_size, shuffle = True)
test_loader = DataLoader(test_trj, batch_size=batch_size, shuffle = True)
history = []

for i in tqdm.tqdm(range(epochs)):
    statistics = dict()
    statistics['training_loss'] = 0.0
    for d in train_loader:
        z = gcn(d)
        l = loss(z, d.label)
        l.backward()
        optimizer.step()
    statistics['training_loss'] += l.item()
    with torch.no_grad():
        statistics['test_accuracy'] = 0.0
        statistics['test_loss'] = 0.0
        for d in test_loader:
            z = gcn(d)
            l = loss(z, d.label)
            a = ((gcn(d).argmax(1) == d.label).sum())
            statistics['test_accuracy'] += a.item()
            statistics['test_loss'] += l.item()
    statistics['test_accuracy'] /= len(test_trj)
    statistics['test_loss'] /= ((len(test_trj) // batch_size) + 1)
    
    history.append(statistics)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 15/15 [22:19<00:00, 89.28s/it]


In [46]:
pd.DataFrame.from_records(history)

,training_loss,test_accuracy,test_loss
0,0.874584,0.5,0.771043
1,0.938262,0.5,0.811099
2,0.563262,0.5,0.811099
3,0.813262,0.5,0.811099
4,0.563262,0.5,0.811099
5,0.938262,0.5,0.811099
6,1.063262,0.5,0.811099
7,0.563262,0.5,0.811099
8,0.563262,0.5,0.811099
9,0.938262,0.5,0.811099


In [ ]:
CHANGE: train test separated, one per each
    training_loss	test_accuracy	test_loss
0	1.313262	0.5	0.811099
1	1.313262	0.5	0.811099
2	1.313262	0.5	0.811099
3	1.313262	0.5	0.811099
4	1.313262	0.5	0.811099
5	1.313262	0.5	0.811099
6	1.313262	0.5	0.811099
7	1.313262	0.5	0.811099
8	1.313262	0.5	0.811099
9	1.313262	0.5	0.811099

CHANGE: epochs=5, train (two simulations) test (one)
	training_loss	test_accuracy	test_loss
0	1.313262	0.5	0.811099
1	1.313262	0.5	0.811099
2	1.313262	0.5	0.811099
3	1.313262	0.5	0.811099
4	1.313262	0.5	0.811099

CHANGE: shuffle=True
    training_loss	test_accuracy	test_loss
0	0.720491	0.500000	0.692701
1	0.698511	0.500000	0.691488
2	0.664993	0.500000	0.692985
3	0.656365	0.499667	0.699392
4	0.519841	0.496667	0.722976

CHANGE: selection_string="name CA N C"
	training_loss	test_accuracy	test_loss
0	0.693184	0.500000	0.691357
1	0.696062	0.500000	0.693620
2	0.718323	0.500000	0.692287
3	0.685770	0.500000	0.691166
4	0.704237	0.499667	0.697881

CHANGE: learning rate
training_loss	test_accuracy	test_loss
0	0.693684	0.5	0.691884
1	0.691864	0.5	0.691319
2	0.693380	0.5	0.691534
3	0.702368	0.5	0.691375
4	0.696929	0.5	0.691310
5	0.688399	0.5	0.691349
6	0.693166	0.5	0.691321
7	0.694098	0.5	0.691311
8	0.697585	0.5	0.691319
9	0.693385	0.5	0.691304
